In [ ]:
import numpy as np
import soundfile as sf
import scipy.signal as sig
import matplotlib.pyplot as plt
import scipy


# ---------------------------
# Loading / basic prep
# ---------------------------
def load_mono(path, target_sr=None):
    y, sr = sf.read(path, always_2d=False)
    if y.ndim > 1:
        y = y.mean(axis=1)
    y = y.astype(np.float32)

    if target_sr is not None and sr != target_sr:
        g = np.gcd(sr, target_sr)
        up = target_sr // g
        down = sr // g
        y = sig.resample_poly(y, up, down).astype(np.float32)
        sr = target_sr

    # normalize to avoid dumb scale issues
    y = y / (np.max(np.abs(y)) + 1e-12)
    return y, sr


def save_wav(path, y, sr):
    sf.write(path, np.clip(y, -1.0, 1.0), sr)


# ---------------------------
# Delay estimation + alignment
# ---------------------------
def estimate_delay_samples(x, d, max_lag=4000):
    """
    Cross-correlation lag estimate.
    Returns lag where positive means x lags behind d and should be advanced.
    """
    n = min(len(x), len(d))
    x0 = x[:n]
    d0 = d[:n]

    # light highpass to reduce DC / slow drift effects
    b, a = sig.butter(2, 50, btype="highpass", fs=48000)
    try:
        x0 = sig.filtfilt(b, a, x0)
        d0 = sig.filtfilt(b, a, d0)
    except Exception:
        pass

    corr = sig.correlate(d0, x0, mode="full", method="fft")
    mid = len(corr) // 2
    lo = mid - max_lag
    hi = mid + max_lag + 1
    seg = corr[lo:hi]
    best = np.argmax(seg) + lo
    lag = best - mid
    return int(lag)


def apply_delay(x, lag):
    if lag == 0:
        return x
    if lag > 0:
        # x is delayed -> advance (drop first lag samples)
        return x[lag:]
    else:
        # x is early -> pad
        return np.pad(x, (abs(lag), 0))
    
def bandpass(y, sr, lo, hi, order=4):
    sos = scipy.signal.butter(order, [lo, hi], btype="bandpass", fs=sr, output="sos")
    return scipy.signal.sosfiltfilt(sos, y)

def estimate_delay_samples(x, d, sr, band=(14000, 18000), max_lag=5000):
    # band-limit first so correlation locks onto air components, not random junk
    x_f = bandpass(x, sr, band[0], band[1])
    d_f = bandpass(d, sr, band[0], band[1])

    corr = scipy.signal.correlate(d_f, x_f, mode="full", method="fft")
    mid = len(corr) // 2
    seg = corr[mid - max_lag : mid + max_lag + 1]
    lag = int(np.argmax(seg) - max_lag)
    return lag


# ---------------------------
# NLMS Adaptive Noise Canceller
# ---------------------------
def nlms_anc(x, d, L=512, mu=0.5, eps=1e-8):
    n = min(len(x), len(d))
    w = np.zeros(L, dtype=np.float32)
    y = np.zeros(n, dtype=np.float32)
    e = np.zeros(n, dtype=np.float32)

    xpad = np.zeros(L - 1 + n, dtype=np.float32)
    xpad[L-1:] = x[:n].astype(np.float32)
    d = d[:n].astype(np.float32)

    for i in range(n):
        xvec = xpad[i:i+L][::-1]
        y_i = np.dot(w, xvec)
        e_i = d[i] - y_i
        w += (mu / (np.dot(xvec, xvec) + eps)) * e_i * xvec
        y[i] = y_i
        e[i] = e_i

    return e, y


# ---------------------------
# Metrics / inspection
# ---------------------------
def erle_db(d, e, sr, t0_s, t1_s):
    """
    ERLE on a segment where milling is absent (air-only).
    Higher is better.
    """
    i0 = int(t0_s * sr)
    i1 = int(t1_s * sr)
    dseg = d[i0:i1]
    eseg = e[i0:i1]
    num = np.mean(dseg**2) + 1e-12
    den = np.mean(eseg**2) + 1e-12
    return 10.0 * np.log10(num / den)


def coherence_stats(x, d, e, sr, fmin=200, fmax=12000):
    f, c_before = sig.coherence(x, d, fs=sr, nperseg=4096)
    _, c_after  = sig.coherence(x, e, fs=sr, nperseg=4096)
    band = (f >= fmin) & (f <= min(fmax, sr/2 - 1))
    return float(np.mean(c_before[band])), float(np.mean(c_after[band])), f, c_before, c_after

def mean_coherence_in_band(x, y, sr, lo, hi):
    f, cxy = scipy.signal.coherence(x, y, fs=sr, nperseg=4096)
    m = (f >= lo) & (f <= hi)
    return float(np.mean(cxy[m])), f, cxy

def plot_spectrogram(
    y, sr, title,
    nperseg=2048, noverlap=1536,
    fmax=20000,
    vmin=None, vmax=None
):
    f, t, S = sig.spectrogram(
        y, fs=sr, window="hann", nperseg=nperseg, noverlap=noverlap,
        scaling="spectrum", mode="magnitude"
    )
    S_db = 20 * np.log10(S + 1e-8)

    fmax_eff = min(fmax, sr / 2)
    mask = f <= fmax_eff
    f = f[mask]
    S_db = S_db[mask, :]

    plt.figure(figsize=(12, 4))
    plt.pcolormesh(t, f, S_db, shading="auto", vmin=vmin, vmax=vmax, cmap="magma")
    plt.ylim(0, fmax_eff)
    plt.xlabel("Time [s]")
    plt.ylabel("Freq [Hz]")
    plt.title(title)
    plt.colorbar(label="Magnitude [dB]")
    plt.tight_layout()


def plot_wave_snip(d, e, sr, t0=1.0, dur=0.05):
    i0 = int(t0 * sr)
    i1 = int((t0 + dur) * sr)
    tt = np.arange(i0, i1) / sr
    plt.figure(figsize=(12, 3))
    plt.plot(tt, d[i0:i1], label="primary (air+milling)")
    plt.plot(tt, e[i0:i1], label="cleaned (milling est.)", alpha=0.85)
    plt.legend()
    plt.title("Waveform snippet")
    plt.tight_layout()


def plot_coherence(f, c_before, c_after):
    plt.figure(figsize=(12, 3))
    plt.semilogx(f, c_before, label="coherence(ref, primary)")
    plt.semilogx(f, c_after,  label="coherence(ref, cleaned)")
    plt.ylim(0, 1.05)
    plt.xlabel("Frequency [Hz]")
    plt.ylabel("Coherence")
    plt.title("Coherence")
    plt.legend()
    plt.tight_layout()


In [ ]:
if __name__ == "__main__":
    # air-dominant mic (reference)
    ref_path = "4_012mm_front_up.flac"
    # milling-dominant mic (primary)
    pri_path = "4_012mm_front_down.flac"

    # Keep original SR (192 kHz)
    d_primary, sr = load_mono(pri_path, target_sr=192000)
    x_ref, _      = load_mono(ref_path, target_sr=sr)

    # --- parameters (tuned for your pair) ---
    LAG_BAND = (14000, 18000)
    MAX_LAG_SAMPLES = 5000

    ANC_BAND = (5000, 10000)      # good default
    # ANC_BAND = (14000, 18000)   # narrower + safer

    L = 512
    MU = 0.5
    EPS = 1e-8

    # --- align channels ---
    lag = estimate_delay_samples(x_ref, d_primary, sr=sr, band=LAG_BAND, max_lag=MAX_LAG_SAMPLES)
    print(f"Estimated lag: {lag} samples = {lag/sr:.6f} s")

    if lag > 0:
        x_ref = x_ref[lag:]
        d_primary = d_primary[:len(x_ref)]
    elif lag < 0:
        d_primary = d_primary[-lag:]
        x_ref = x_ref[:len(d_primary)]

    # safety trim
    n = min(len(x_ref), len(d_primary))
    x_ref = x_ref[:n]
    d_primary = d_primary[:n]

    # --- band-limited adaptation (IMPORTANT) ---
    x_adapt = bandpass(x_ref, sr, *ANC_BAND)
    d_adapt = bandpass(d_primary, sr, *ANC_BAND)

    # --- ANC (NLMS) on band-limited signals ---
    e_band, y_band = nlms_anc(x_adapt, d_adapt, L=L, mu=MU, eps=EPS)
    
    # subtract estimated noise from the fullband primary
    cleaned = d_primary - y_band

    save_wav("cleaned_milling_est.wav", cleaned, sr)
    save_wav("estimated_air_noise_band.wav", y_band, sr)

    # --- metrics (use band-limited signals for meaningful numbers) ---
    # ERLE needs an air-only segment (if you genuinely have one)
    try:
        erle = erle_db(d_adapt, e_band, sr, t0_s=2.0, t1_s=11.5)
        print(f"ERLE on air-only segment (band): {erle:.2f} dB")
    except Exception:
        print("ERLE skipped (pick a known air-only segment).")

    cb_mean, f, c_before = mean_coherence_in_band(x_adapt, d_adapt, sr, *ANC_BAND)
    ca_mean, _, c_after  = mean_coherence_in_band(x_adapt, e_band, sr, *ANC_BAND)
    print("Mean coherence in ANC_BAND before/after:", cb_mean, ca_mean)

    plot_coherence(f, c_before, c_after)
    
    # --- plots ---
    plot_wave_snip(d_primary, cleaned, sr, t0=1.0, dur=0.05)
    plot_spectrogram(d_primary, sr, "Primary", fmax=12000, vmin=-120, vmax=-20)
    plot_spectrogram(cleaned,  sr, "Cleaned", fmax=12000, vmin=-120, vmax=-20)
    plot_coherence(f, cb, ca)
    plt.show()